# Qwen3 Associative Retrieval Position-Rescue Mechanism
## v0.6.0 — frozen 4K/8K causal mechanism pilot

This notebook consumes the **frozen v0.5.0.1 calibration** and runs the central causal test.

Frozen settings expected from calibration:

- task: associative retrieval (`R1`);
- `N_RECORDS = 32`;
- canonical lengths: 4K and 8K;
- filler fragmentation strength:
  - English filler: `keep=2`;
  - Turkish filler: `keep=5`.

The notebook verifies those values from `v050_audit.json` before doing any model evaluation.

## Central causal contrast

For each exact same prompt:

1. **canonical_normal**  
   Canonical tokenization, ordinary sequential positions.

2. **filler_frag_normal**  
   Fragment only the irrelevant filler; use ordinary sequential positions.  
   This inflates token distance from evidence to query/readout.

3. **filler_frag_block_affine**  
   Use the **same fragmented token IDs** as condition 2, but compress the entire filler block uniformly back into its canonical positional span.

4. **filler_frag_fractional**  
   Use the **same fragmented token IDs** as conditions 2–3, but place fragments inside the positions of their original canonical source tokens:
   `p + j/m`.

Thus:

- `filler_frag_normal − canonical_normal` estimates the total filler-fragmentation penalty;
- `filler_frag_normal − filler_frag_fractional` is the source-aligned positional rescue;
- `filler_frag_normal − filler_frag_block_affine` is a generic positional-compression rescue;
- `filler_frag_block_affine − filler_frag_fractional` asks whether preserving local source-token structure adds benefit beyond restoring the same global span.

Because the fragmented conditions have **identical token IDs**, any difference among the three fragmented-position conditions is caused by the position intervention.

## Confirmatory separation from calibration

The mechanism run uses **16 new holdout semantic seeds** that were not used to select record count or fragmentation strength.

Gold digits and target-record positions are jointly balanced.

## Primary outcome

**Forced-choice NLL over the legal stored values `{1,2,3,4}`**.

Secondary outcomes:

- full-vocabulary gold-token NLL;
- unrestricted full-vocabulary first-token argmax;
- accuracy.

No parameter is retuned on mechanism outcomes.

> This notebook tests the positional component first. If it yields a clean rescue, the next stage can add evidence-locus fragmentation to estimate the residual representation component without changing this central analysis.

> Public reproduction copy: calibration values are frozen explicitly; calibration scans and development/debugging history are not required to reproduce the paper's reported experiment.

## 0. T4 runtime

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

## 1. Drive, frozen calibration, and configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, shutil

PROJECT_DIR = Path('/content/drive/MyDrive/qwen3_assoc_mechanism_v060')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = PROJECT_DIR / 'v060_mechanism_4k8k.jsonl'
AUDIT_PATH = PROJECT_DIR / 'v060_audit.json'
ZIP_PATH = PROJECT_DIR / 'fragburden0.3.0.zip'
REPO_DIR = PROJECT_DIR / 'fragburden_src' / 'fragburden'

MODEL_NAME = 'Qwen/Qwen3-4B'
TRANSFORMERS_VERSION = '5.15.1'
TASK_VERSION = 'symbolic_assoc_r1_v060'
SCORER_VERSION = 'assoc_chunked_floatpos_v060'
POSITION_INTERVENTION = 'fractional_vs_block_affine_v2'
PREFILL_CHUNK = 512
TARGET_LENGTHS = [4096, 8192]
FILLER_LANGUAGES = ['en', 'tr']
SEMANTIC_SEEDS = [1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1015]
POSITION_RANK_SET = [0, 2, 5, 7]
BOOTSTRAP_REPS = 10000
BOOTSTRAP_SEED = 20260828

# Frozen during outcome-independent calibration and reported in the paper.
N_RECORDS = 32
FRAG_KEEP_BY_LANGUAGE = {'en': 2, 'tr': 5}
previous = None

print('FROZEN PAPER SETTINGS')
print('N_RECORDS =', N_RECORDS)
print('FRAG_KEEP_BY_LANGUAGE =', FRAG_KEEP_BY_LANGUAGE)
print('SEMANTIC_SEEDS =', SEMANTIC_SEEDS)


## 2. Install `fragburden` and validated Transformers

In [ ]:
import subprocess, sys

if not ZIP_PATH.exists():
    repo_zip = Path('/content/token-distance-positional-geometry/code/fragburden0.3.0.zip')
    if repo_zip.exists():
        shutil.copy2(repo_zip, ZIP_PATH)
        print('Copied fragburden from cloned repository.')
    else:
        from google.colab import files
        uploaded = files.upload()
        zip_names = [n for n in uploaded if n.endswith('.zip')]
        if not zip_names:
            raise RuntimeError('Upload code/fragburden0.3.0.zip from the repository.')
        shutil.copy2(Path(zip_names[0]), ZIP_PATH)

extract_root = PROJECT_DIR / 'fragburden_src'
if extract_root.exists(): shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(str(ZIP_PATH), str(extract_root))
assert REPO_DIR.exists() and (REPO_DIR / 'pyproject.toml').exists()
subprocess.check_call([sys.executable,'-m','pip','install','-e',f'{REPO_DIR}[gpu,analysis]'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','-U',f'transformers=={TRANSFORMERS_VERSION}','accelerate','bitsandbytes'])
print('Installation complete.')


## 3. Imports and Qwen3-4B

In [ ]:
import hashlib, random, math, inspect, sys, gc, time
from dataclasses import dataclass

import numpy as np
import pandas as pd

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import torch
import torch.nn.functional as F
import transformers

import fragburden
from fragburden.dial import FragmentationDial, verify_monotone
from fragburden.testing import sample_texts

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DynamicCache,
)

assert transformers.__version__ == TRANSFORMERS_VERSION
assert torch.cuda.is_available()

print('fragburden:', fragburden.__file__)
print('transformers:', transformers.__version__)
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

load_kwargs = dict(
    quantization_config=bnb,
    device_map='auto',
    attn_implementation='sdpa',
)

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.float16,
        **load_kwargs,
    )
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        **load_kwargs,
    )

model.eval()

if hasattr(model.config, 'use_cache'):
    model.config.use_cache = False

backbone = model.model
device = next(model.parameters()).device

if 'position_ids' not in inspect.signature(backbone.forward).parameters:
    raise RuntimeError('Backbone does not expose position_ids.')

print('Loaded:', MODEL_NAME)

## 4. Fragmentation-dial capability gate

In [ ]:
dial = FragmentationDial.from_transformers(
    tokenizer,
    name=MODEL_NAME,
)

cap = dial.capability()
print(json.dumps(cap, indent=2))

dial_gate = verify_monotone(
    dial,
    sample_texts('en') + sample_texts('tr'),
    levels=(100, 50, 25, 10, 5, 2, 1),
)

print(json.dumps(dial_gate, indent=2))

if not cap.get('usable_for_arm_a'):
    raise RuntimeError('No usable fragmentation dial.')

if (
    not dial_gate['monotone_nondecreasing']
    or not dial_gate['all_in_vocab']
):
    raise RuntimeError('Fragmentation dial failed.')

d100 = dial.at(100)

DIALS = {
    lang: dial.at(FRAG_KEEP_BY_LANGUAGE[lang])
    for lang in FILLER_LANGUAGES
}

print('Canonical fingerprint:', d100.fingerprint)
print(
    'Fragmented fingerprints:',
    {
        lang: DIALS[lang].fingerprint
        for lang in FILLER_LANGUAGES
    },
)

print('DIAL GATE PASSED.')

## 5. Official non-thinking chat shell and digit candidates

In [ ]:
DIGITS = list('1234')

def _normalize_input_ids(obj):
    if hasattr(obj, 'keys') and 'input_ids' in obj:
        obj = obj['input_ids']

    if torch.is_tensor(obj):
        obj = obj.detach().cpu().tolist()

    if (
        isinstance(obj, (list, tuple))
        and len(obj) == 1
        and isinstance(obj[0], (list, tuple))
    ):
        obj = obj[0]

    return [int(x) for x in obj]

def apply_chat_ids(user_content):
    out = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': user_content}],
        tokenize=True,
        return_dict=True,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    return _normalize_input_ids(out)

def render_chat(user_content):
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': user_content}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

DIGIT_IDS = {}

for d in DIGITS:
    ids = tokenizer(
        d,
        add_special_tokens=False,
    )['input_ids']

    if len(ids) != 1:
        raise RuntimeError(f'{d} is not a one-token candidate: {ids}')

    DIGIT_IDS[d] = int(ids[0])

if len(set(DIGIT_IDS.values())) != 4:
    raise RuntimeError('Digit tokens are not distinct.')

print('Digit IDs:', DIGIT_IDS)

## 6. Exact official-chat shell decomposition

In [ ]:
SHELL_MARKER = 'UNIQUE_USER_CONTENT_MARKER_V060_91C7F2'

rendered_marker = render_chat(SHELL_MARKER)

if rendered_marker.count(SHELL_MARKER) != 1:
    raise RuntimeError('Could not uniquely locate user content.')

marker_idx = rendered_marker.index(SHELL_MARKER)

CHAT_PREFIX_TEXT = rendered_marker[:marker_idx]
CHAT_SUFFIX_TEXT = rendered_marker[
    marker_idx + len(SHELL_MARKER):
]

CHAT_PREFIX_IDS = tokenizer(
    CHAT_PREFIX_TEXT,
    add_special_tokens=False,
)['input_ids']

CHAT_SUFFIX_IDS = tokenizer(
    CHAT_SUFFIX_TEXT,
    add_special_tokens=False,
)['input_ids']

def canonical_user_ids(user_content):
    ids_tok = tokenizer(
        user_content,
        add_special_tokens=False,
    )['input_ids']

    ids_dial = d100.pieces_to_ids(
        d100.encode_pieces(user_content)
    )

    if ids_tok != ids_dial:
        raise RuntimeError(
            'fragburden canonical IDs disagree with tokenizer.'
        )

    return ids_dial

def verify_chat_shell(user_content):
    user_ids = canonical_user_ids(user_content)

    reconstructed = (
        CHAT_PREFIX_IDS
        + user_ids
        + CHAT_SUFFIX_IDS
    )

    official = apply_chat_ids(user_content)

    if reconstructed != official:
        raise RuntimeError('Chat-shell reconstruction failed.')

    return {
        'official_ids': official,
        'canonical_user_ids': user_ids,
    }

print('Prefix tokens:', len(CHAT_PREFIX_IDS))
print('Suffix tokens:', len(CHAT_SUFFIX_IDS))
print('Suffix tail:', repr(CHAT_SUFFIX_TEXT[-120:]))
print('Chat shell ready.')

# Holdout associative-retrieval instances

## 7. Balanced semantic seeds and target-record positions

For 16 holdout instances:

- each gold digit occurs exactly four times;
- each gold digit appears once at each of four position ranks spanning the evidence list.

This breaks the gold/position association that affected some earlier pilots.

In [ ]:
@dataclass
class CharSpan:
    start: int
    end: int
    label: str

@dataclass
class AssocInstance:
    filler_language: str
    semantic_seed: int
    trial_index: int
    target_chat_tokens: int
    n_records: int
    user_content: str
    spans: list
    target_key: str
    gold_digit: str
    target_record_position: int
    evidence_lines: list
    filler_lines: list
    query_lines: list

def stable_int(*parts):
    blob = '|'.join(map(str, parts)).encode('utf-8')

    return int.from_bytes(
        hashlib.sha256(blob).digest()[:8],
        'big',
    )

def stable_rng(*parts):
    return random.Random(stable_int(*parts))

def make_key(semantic_seed, slot, n_records):
    value = (
        stable_int(
            'key_v060',
            semantic_seed,
            slot,
            n_records,
        )
        % 90000
        + 10000
    )

    return f'K{value}'

def gold_for_trial(trial_index):
    return DIGITS[trial_index % 4]

def position_rank_for_trial(trial_index):
    gold_index = trial_index % 4
    replicate = trial_index // 4

    return POSITION_RANK_SET[
        (replicate + gold_index)
        % len(POSITION_RANK_SET)
    ]

def target_position_for_trial(trial_index, n_records):
    rank = position_rank_for_trial(trial_index)

    return int(
        round(
            rank
            * (n_records - 1)
            / 7
        )
    )

FILLER_BANK = {
    'en': {
        'subjects': [
            'The archivist',
            'The technician',
            'The editor',
            'The visitor',
            'The clerk',
            'The gardener',
        ],
        'verbs': [
            'reviewed',
            'checked',
            'organized',
            'recorded',
            'sorted',
            'noted',
        ],
        'objects': [
            'a routine report',
            'the daily schedule',
            'a short memo',
            'the supply list',
            'a maintenance note',
            'the meeting agenda',
        ],
        'settings': [
            'near the window',
            'in the quiet office',
            'before lunch',
            'during the afternoon',
            'at the front desk',
            'after the meeting',
        ],
    },
    'tr': {
        'subjects': [
            'Arşiv görevlisi',
            'Teknisyen',
            'Editör',
            'Ziyaretçi',
            'Görevli',
            'Bahçıvan',
        ],
        'verbs': [
            'inceledi',
            'kontrol etti',
            'düzenledi',
            'kaydetti',
            'sıraladı',
            'not etti',
        ],
        'objects': [
            'sıradan bir raporu',
            'günlük programı',
            'kısa bir notu',
            'malzeme listesini',
            'bakım notunu',
            'toplantı gündemini',
        ],
        'settings': [
            'pencerenin yanında',
            'sessiz ofiste',
            'öğle yemeğinden önce',
            'öğleden sonra',
            'ön masada',
            'toplantıdan sonra',
        ],
    },
}

def filler_sentence(language, idx, semantic_seed):
    bank = FILLER_BANK[language]

    x = idx + 7919 * semantic_seed

    subject = bank['subjects'][x % len(bank['subjects'])]
    x //= len(bank['subjects'])

    verb = bank['verbs'][x % len(bank['verbs'])]
    x //= len(bank['verbs'])

    obj = bank['objects'][x % len(bank['objects'])]
    x //= len(bank['objects'])

    setting = bank['settings'][x % len(bank['settings'])]

    if language == 'en':
        return f'{subject} {verb} {obj} {setting}.'

    return f'{subject} {setting} {obj} {verb}.'

def make_records(semantic_seed, trial_index, n_records):
    if n_records % 4 != 0:
        raise ValueError('n_records must be divisible by 4.')

    gold = gold_for_trial(trial_index)

    values = DIGITS * (n_records // 4)

    gold_idx = values.index(gold)

    values[0], values[gold_idx] = (
        values[gold_idx],
        values[0],
    )

    records = [
        (
            make_key(
                semantic_seed,
                slot,
                n_records,
            ),
            values[slot],
        )
        for slot in range(n_records)
    ]

    target = records[0]

    others = records[1:]

    stable_rng(
        'records_v060',
        semantic_seed,
        trial_index,
        n_records,
    ).shuffle(others)

    pos = target_position_for_trial(
        trial_index,
        n_records,
    )

    ordered = others[:]
    ordered.insert(pos, target)

    if len(ordered) != n_records:
        raise RuntimeError('Record count changed.')

    return ordered, target[0], gold, pos

def base_evidence_query(
    semantic_seed,
    trial_index,
    n_records,
):
    records, target_key, gold, pos = make_records(
        semantic_seed,
        trial_index,
        n_records,
    )

    evidence = [
        'Task: look up the value for the queried key.',
        'Each evidence line has the form KEY=VALUE.',
        'Return only the VALUE associated with the queried KEY.',
        'Do not explain or repeat the key.',
        'EVIDENCE:',
    ]

    evidence += [
        f'{key}={value}'
        for key, value in records
    ]

    query = [
        f'QUERY KEY: {target_key}',
        'Answer with one digit only:',
    ]

    return evidence, query, target_key, gold, pos

def compose_user_content(
    evidence,
    filler,
    query,
):
    evidence_text = '\n'.join(evidence)
    filler_text = '\n'.join(filler)
    query_text = '\n'.join(query)

    if not filler_text:
        user_content = (
            evidence_text
            + '\n'
            + query_text
        )

        query_start = len(evidence_text) + 1

        spans = [
            CharSpan(
                0,
                query_start,
                'evidence',
            ),
            CharSpan(
                query_start,
                len(user_content),
                'query',
            ),
        ]

        return user_content, spans

    user_content = (
        evidence_text
        + '\n'
        + filler_text
        + '\n'
        + query_text
    )

    filler_start = len(evidence_text) + 1
    query_start = (
        filler_start
        + len(filler_text)
        + 1
    )

    # Include boundary newlines in the preceding blocks so the three
    # semantic regions cover the complete character string contiguously.
    spans = [
        CharSpan(
            0,
            filler_start,
            'evidence',
        ),
        CharSpan(
            filler_start,
            query_start,
            'filler',
        ),
        CharSpan(
            query_start,
            len(user_content),
            'query',
        ),
    ]

    return user_content, spans

def build_instance(
    filler_language,
    semantic_seed,
    trial_index,
    target_chat_tokens,
    n_records=N_RECORDS,
):
    (
        evidence,
        query,
        target_key,
        gold,
        pos,
    ) = base_evidence_query(
        semantic_seed,
        trial_index,
        n_records,
    )

    filler = []

    while True:
        user_content, spans = compose_user_content(
            evidence,
            filler,
            query,
        )

        full_len = len(
            apply_chat_ids(user_content)
        )

        if full_len >= target_chat_tokens:
            break

        filler.append(
            filler_sentence(
                filler_language,
                len(filler),
                semantic_seed,
            )
        )

        if len(filler) > 5000:
            raise RuntimeError('Filler loop failed.')

    verify_chat_shell(user_content)

    return AssocInstance(
        filler_language=filler_language,
        semantic_seed=semantic_seed,
        trial_index=trial_index,
        target_chat_tokens=target_chat_tokens,
        n_records=n_records,
        user_content=user_content,
        spans=spans,
        target_key=target_key,
        gold_digit=gold,
        target_record_position=pos,
        evidence_lines=evidence,
        filler_lines=filler,
        query_lines=query,
    )

print('Holdout task generator ready.')

## 8. Gold/position balancing audit

In [ ]:
balance_rows = []

for trial_index, semantic_seed in enumerate(SEMANTIC_SEEDS):
    balance_rows.append({
        'trial_index': trial_index,
        'semantic_seed': semantic_seed,
        'gold_digit': gold_for_trial(trial_index),
        'position_rank': position_rank_for_trial(trial_index),
        'target_record_position': target_position_for_trial(
            trial_index,
            N_RECORDS,
        ),
    })

balance = pd.DataFrame(balance_rows)
display(balance)

gold_position = pd.crosstab(
    balance['gold_digit'],
    balance['position_rank'],
)

display(gold_position)

if not (balance['gold_digit'].value_counts().sort_index() == 4).all():
    raise RuntimeError('Gold digits are not exactly balanced.')

if not (gold_position == 1).all().all():
    raise RuntimeError(
        'Each gold digit must appear exactly once at every position rank.'
    )

print('GOLD/POSITION BALANCE GATE PASSED.')

# Byte-identical filler retokenization and position interventions

## 9. Source-aligned fractional positions and block-affine control

In [ ]:
def span_label(start, end, spans):
    best_label = None
    best_overlap = 0

    for s in spans:
        overlap = max(
            0,
            min(end, s.end)
            - max(start, s.start),
        )

        if overlap > best_overlap:
            best_overlap = overlap
            best_label = s.label

    if best_label is None:
        raise RuntimeError(
            f'Pretoken chunk [{start},{end}) did not overlap any semantic span.'
        )

    return best_label

def _piece_intervals(pieces):
    intervals = []
    pos = 0

    for piece in pieces:
        intervals.append(
            (pos, pos + len(piece))
        )
        pos += len(piece)

    return intervals, pos

def _fractional_local_positions(
    canonical_pieces,
    fragmented_pieces,
):
    if ''.join(canonical_pieces) != ''.join(fragmented_pieces):
        raise ValueError(
            'Canonical/fragmented pieces reconstruct different text.'
        )

    c_int, c_total = _piece_intervals(canonical_pieces)
    f_int, f_total = _piece_intervals(fragmented_pieces)

    if c_total != f_total:
        raise ValueError(
            'Canonical/fragmented character extents disagree.'
        )

    assignment = []
    c_idx = 0

    for fa, fb in f_int:
        while (
            c_idx + 1 < len(c_int)
            and c_int[c_idx][1] <= fa
        ):
            c_idx += 1

        ca, cb = c_int[c_idx]

        if not (
            ca <= fa
            and fb <= cb
        ):
            raise ValueError(
                'A fragmented token crossed a canonical-token boundary.'
            )

        assignment.append(c_idx)

    groups = {}

    for fine_idx, coarse_idx in enumerate(assignment):
        groups.setdefault(
            coarse_idx,
            [],
        ).append(fine_idx)

    pos = [None] * len(fragmented_pieces)

    for coarse_idx, fine_indices in groups.items():
        m = len(fine_indices)

        for rank, fine_idx in enumerate(fine_indices):
            pos[fine_idx] = (
                coarse_idx
                + rank / m
            )

    arr = np.asarray(
        pos,
        dtype=np.float32,
    )

    if (
        len(arr) > 1
        and not np.all(np.diff(arr) > 0)
    ):
        raise ValueError(
            'Fractional local positions are not strictly increasing.'
        )

    return arr.tolist()

def build_user_encoding(
    user_content,
    spans,
    fragment_target=None,
    dfrag=None,
):
    canonical_ids = []
    mixed_ids = []

    source_positions = []
    token_labels = []
    canonical_positions_for_mixed = []

    canonical_cursor = 0

    for chunk, (a, b) in d100.pre_tokenize(user_content):
        label = span_label(
            a,
            b,
            spans,
        )

        c_pieces = d100.encode_chunk_pieces(chunk)
        c_ids = d100.pieces_to_ids(c_pieces)

        should_fragment = (
            fragment_target == label
        )

        if should_fragment:
            if dfrag is None:
                raise RuntimeError(
                    'Fragment target requested without a fragmentation dial.'
                )

            m_pieces = dfrag.encode_chunk_pieces(chunk)
            m_ids = d100.pieces_to_ids(m_pieces)

            local_frac = _fractional_local_positions(
                c_pieces,
                m_pieces,
            )

            src = [
                canonical_cursor + x
                for x in local_frac
            ]

            coarse_abs = [
                canonical_cursor
                + int(
                    math.floor(x + 1e-8)
                )
                for x in local_frac
            ]

        else:
            m_ids = c_ids

            src = [
                float(x)
                for x in range(
                    canonical_cursor,
                    canonical_cursor + len(c_ids),
                )
            ]

            coarse_abs = list(
                range(
                    canonical_cursor,
                    canonical_cursor + len(c_ids),
                )
            )

        canonical_ids.extend(c_ids)
        mixed_ids.extend(m_ids)
        source_positions.extend(src)
        canonical_positions_for_mixed.extend(coarse_abs)
        token_labels.extend([label] * len(m_ids))

        canonical_cursor += len(c_ids)

    whole_canonical = canonical_user_ids(user_content)

    if canonical_ids != whole_canonical:
        raise RuntimeError(
            'Chunkwise canonical user encoding differs from whole-user encoding.'
        )

    decoded_user = tokenizer.decode(
        mixed_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    if decoded_user != user_content:
        mismatch = next(
            (
                i
                for i, (x, y) in enumerate(
                    zip(decoded_user, user_content)
                )
                if x != y
            ),
            None,
        )

        raise RuntimeError(
            'Fragmented user IDs do not decode to identical user text; '
            f'mismatch={mismatch}'
        )

    normal = np.arange(
        len(mixed_ids),
        dtype=np.float32,
    )

    fractional = np.asarray(
        source_positions,
        dtype=np.float32,
    )

    block_affine = fractional.copy()

    if fragment_target is not None:
        labels_np = np.asarray(token_labels)
        idx = np.where(
            labels_np == fragment_target
        )[0]

        if not len(idx):
            raise RuntimeError(
                f'No tokens labeled {fragment_target}.'
            )

        if not np.all(np.diff(idx) == 1):
            raise RuntimeError(
                f'{fragment_target} token region is not contiguous.'
            )

        block_start = float(
            min(
                canonical_positions_for_mixed[i]
                for i in idx
            )
        )

        block_last = float(
            max(
                canonical_positions_for_mixed[i]
                for i in idx
            )
        )

        block_end_exclusive = (
            block_last + 1.0
        )

        n_block = len(idx)

        block_affine[idx] = (
            block_start
            + np.arange(
                n_block,
                dtype=np.float32,
            )
            * (
                block_end_exclusive
                - block_start
            )
            / n_block
        )

    for name, p in [
        ('normal', normal),
        ('block_affine', block_affine),
        ('fractional', fractional),
    ]:
        if (
            len(p) > 1
            and not np.all(np.diff(p) > 0)
        ):
            bad = np.where(
                np.diff(p) <= 0
            )[0][:10]

            raise RuntimeError(
                f'{name} user positions are not strictly increasing; '
                f'bad={bad.tolist()}'
            )

    return {
        'canonical_ids': canonical_ids,
        'mixed_ids': mixed_ids,
        'labels': token_labels,
        'normal_positions': normal,
        'block_affine_positions': block_affine,
        'fractional_positions': fractional,
        'canonical_user_len': len(canonical_ids),
        'realized_user_len': len(mixed_ids),
    }

print('User retokenization utilities ready.')

## 10. Full-chat assembly and geometry diagnostics

In [ ]:
def hash_ids(ids):
    return hashlib.sha256(
        np.asarray(
            ids,
            dtype=np.int64,
        ).tobytes()
    ).hexdigest()[:24]

def hash_positions(pos):
    return hashlib.sha256(
        np.asarray(
            pos,
            dtype=np.float32,
        ).tobytes()
    ).hexdigest()[:24]

def assemble_chat_encoding(
    inst,
    fragment_target=None,
):
    shell = verify_chat_shell(
        inst.user_content
    )

    dfrag = (
        DIALS[
            inst.filler_language
        ]
        if fragment_target is not None
        else None
    )

    user = build_user_encoding(
        inst.user_content,
        inst.spans,
        fragment_target=fragment_target,
        dfrag=dfrag,
    )

    if user['canonical_ids'] != shell['canonical_user_ids']:
        raise RuntimeError(
            'User canonical IDs disagree with shell audit.'
        )

    prefix_len = len(CHAT_PREFIX_IDS)
    suffix_len = len(CHAT_SUFFIX_IDS)

    canonical_user_len = user['canonical_user_len']

    full_ids = (
        CHAT_PREFIX_IDS
        + user['mixed_ids']
        + CHAT_SUFFIX_IDS
    )

    canonical_full_ids = shell['official_ids']
    canonical_full_len = len(canonical_full_ids)

    normal = np.arange(
        len(full_ids),
        dtype=np.float32,
    )

    prefix_pos = np.arange(
        prefix_len,
        dtype=np.float32,
    )

    suffix_start = (
        prefix_len
        + canonical_user_len
    )

    suffix_pos = np.arange(
        suffix_start,
        suffix_start + suffix_len,
        dtype=np.float32,
    )

    block_affine = np.concatenate(
        [
            prefix_pos,
            prefix_len
            + user['block_affine_positions'],
            suffix_pos,
        ]
    ).astype(np.float32)

    fractional = np.concatenate(
        [
            prefix_pos,
            prefix_len
            + user['fractional_positions'],
            suffix_pos,
        ]
    ).astype(np.float32)

    for name, p in [
        ('normal', normal),
        ('block_affine', block_affine),
        ('fractional', fractional),
    ]:
        if (
            len(p) > 1
            and not np.all(np.diff(p) > 0)
        ):
            bad = np.where(
                np.diff(p) <= 0
            )[0][:10]

            raise RuntimeError(
                f'{name} full-chat positions are not strictly increasing; '
                f'bad={bad.tolist()}'
            )

    labels_np = np.asarray(
        user['labels']
    )

    evidence_idx = np.where(
        labels_np == 'evidence'
    )[0]

    filler_idx = np.where(
        labels_np == 'filler'
    )[0]

    query_idx = np.where(
        labels_np == 'query'
    )[0]

    if not (
        len(evidence_idx)
        and len(filler_idx)
        and len(query_idx)
    ):
        raise RuntimeError(
            'Missing evidence/filler/query token region.'
        )

    def shifted_user_position(
        full_position_array,
        user_index,
    ):
        return float(
            full_position_array[
                prefix_len
                + user_index
            ]
        )

    def gaps(pos):
        evidence_last = shifted_user_position(
            pos,
            int(evidence_idx[-1]),
        )

        filler_first = shifted_user_position(
            pos,
            int(filler_idx[0]),
        )

        filler_last = shifted_user_position(
            pos,
            int(filler_idx[-1]),
        )

        query_first = shifted_user_position(
            pos,
            int(query_idx[0]),
        )

        query_last = shifted_user_position(
            pos,
            int(query_idx[-1]),
        )

        readout = float(pos[-1])

        return {
            'evidence_query': (
                query_first
                - evidence_last
            ),
            'evidence_readout': (
                readout
                - evidence_last
            ),
            'filler_span': (
                filler_last
                - filler_first
            ),
            'query_readout': (
                readout
                - query_last
            ),
        }

    return {
        'ids': full_ids,
        'canonical_full_ids': canonical_full_ids,
        'normal_positions': normal.tolist(),
        'block_affine_positions': block_affine.tolist(),
        'fractional_positions': fractional.tolist(),
        'canonical_full_len': canonical_full_len,
        'realized_full_len': len(full_ids),
        'canonical_user_len': canonical_user_len,
        'realized_user_len': user['realized_user_len'],
        'inflation_full': (
            len(full_ids)
            / canonical_full_len
        ),
        'inflation_user': (
            user['realized_user_len']
            / canonical_user_len
        ),
        'gaps_normal': gaps(normal),
        'gaps_block_affine': gaps(block_affine),
        'gaps_fractional': gaps(fractional),
        'ids_hash': hash_ids(full_ids),
        'normal_position_hash': hash_positions(normal),
        'block_position_hash': hash_positions(block_affine),
        'fractional_position_hash': hash_positions(fractional),
        'prefix_len': prefix_len,
        'suffix_len': suffix_len,
    }

print('Full-chat assembly ready.')

# T4-safe scorer

## 11. Chunked KV prefill with sequential causal mask independent of experimental positions

In [ ]:
def _sequential_chunk_mask(
    start,
    end,
    device,
):
    q = torch.arange(
        start,
        end,
        device=device,
    )[:, None]

    k = torch.arange(
        0,
        end,
        device=device,
    )[None, :]

    return (
        k <= q
    )[None, None, :, :]

def finish_digit_scores(
    h_last,
    gold_digit,
):
    logits = model.lm_head(
        h_last
    ).float()[0]

    candidate_ids = torch.tensor(
        [
            DIGIT_IDS[d]
            for d in DIGITS
        ],
        dtype=torch.long,
        device=logits.device,
    )

    candidate_logits = logits.index_select(
        0,
        candidate_ids,
    )

    forced_logp = F.log_softmax(
        candidate_logits,
        dim=-1,
    )

    forced_prob = forced_logp.exp()

    full_logp = F.log_softmax(
        logits,
        dim=-1,
    )

    gold_idx = DIGITS.index(
        gold_digit
    )

    candidate_pred_idx = int(
        torch.argmax(
            candidate_logits
        ).item()
    )

    candidate_pred = DIGITS[
        candidate_pred_idx
    ]

    full_argmax_id = int(
        torch.argmax(
            logits
        ).item()
    )

    full_argmax_digit = next(
        (
            d
            for d, tok_id in DIGIT_IDS.items()
            if tok_id == full_argmax_id
        ),
        None,
    )

    full_argmax_text = tokenizer.decode(
        [full_argmax_id],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    result = {
        'predicted_digit': candidate_pred,
        'correct': int(
            candidate_pred == gold_digit
        ),
        'forced_choice_nll': float(
            -forced_logp[gold_idx].item()
        ),
        'gold_full_nll': float(
            -full_logp[
                DIGIT_IDS[gold_digit]
            ].item()
        ),
        'gold_candidate_probability': float(
            forced_prob[gold_idx].item()
        ),
        'full_argmax_id': full_argmax_id,
        'full_argmax_text': full_argmax_text,
        'full_argmax_digit': full_argmax_digit,
        'full_argmax_is_gold': int(
            full_argmax_digit == gold_digit
        ),
        'full_argmax_is_legal_digit': int(
            full_argmax_digit is not None
        ),
        'candidate_probabilities': {
            d: float(
                forced_prob[i].item()
            )
            for i, d in enumerate(DIGITS)
        },
    }

    del (
        logits,
        candidate_ids,
        candidate_logits,
        forced_logp,
        forced_prob,
        full_logp,
    )

    return result

def score_chunked(
    input_ids,
    position_ids,
    gold_digit,
    chunk_size=PREFILL_CHUNK,
    position_dtype=torch.float32,
):
    if len(input_ids) != len(position_ids):
        raise ValueError(
            'IDs/positions length mismatch.'
        )

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    ids_all = torch.tensor(
        input_ids,
        dtype=torch.long,
        device=device,
    )

    pos_all = torch.tensor(
        position_ids,
        dtype=position_dtype,
        device=device,
    )

    cache = DynamicCache(
        config=model.config
    )

    last_hidden = None

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    with torch.inference_mode():
        n = ids_all.numel()

        for start in range(
            0,
            n,
            chunk_size,
        ):
            end = min(
                start + chunk_size,
                n,
            )

            ids_chunk = ids_all[
                start:end
            ].unsqueeze(0)

            pos_chunk = pos_all[
                start:end
            ].unsqueeze(0)

            causal = _sequential_chunk_mask(
                start,
                end,
                device,
            )

            out = backbone(
                input_ids=ids_chunk,
                attention_mask={
                    'full_attention': causal
                },
                position_ids=pos_chunk,
                past_key_values=cache,
                use_cache=True,
                return_dict=True,
            )

            cache = out.past_key_values

            if end == n:
                last_hidden = out.last_hidden_state[
                    :, -1, :
                ]

            del (
                ids_chunk,
                pos_chunk,
                causal,
                out,
            )

        scored = finish_digit_scores(
            last_hidden,
            gold_digit,
        )

    torch.cuda.synchronize()

    scored['seconds'] = (
        time.perf_counter()
        - t0
    )

    scored['peak_cuda_gb'] = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

    scored['prefill_chunk'] = chunk_size
    scored['scorer_version'] = SCORER_VERSION

    del (
        ids_all,
        pos_all,
        cache,
        last_hidden,
    )

    gc.collect()
    torch.cuda.empty_cache()

    return scored

def flatten_score(scored):
    out = {
        k: v
        for k, v in scored.items()
        if k != 'candidate_probabilities'
    }

    for d, p in scored[
        'candidate_probabilities'
    ].items():
        out[f'prob_{d}'] = p

    return out

print('Chunked scorer ready.')

## 12. Structural engineering gate at ~1K

This cell validates the intervention machinery, not an effect direction.

It requires:

- exact same decoded user text;
- same fragmented token IDs across normal/block/fractional mappings;
- normal fragmentation increases evidence→readout distance;
- block-affine and fractional mappings restore the canonical global evidence→readout distance;
- block-affine and fractional position arrays are not identical;
- all four scores are finite.

In [ ]:
gate_inst = build_instance(
    filler_language='en',
    semantic_seed=1999,
    trial_index=2,
    target_chat_tokens=1024,
)

gate_can = assemble_chat_encoding(
    gate_inst,
    fragment_target=None,
)

gate_frag = assemble_chat_encoding(
    gate_inst,
    fragment_target='filler',
)

if gate_frag['realized_full_len'] <= gate_can['realized_full_len']:
    raise RuntimeError(
        'Fragmentation did not increase token count.'
    )

can_gap = gate_can['gaps_normal']['evidence_readout']
normal_gap = gate_frag['gaps_normal']['evidence_readout']
block_gap = gate_frag['gaps_block_affine']['evidence_readout']
frac_gap = gate_frag['gaps_fractional']['evidence_readout']

print('Lengths:', {
    'canonical': gate_can['realized_full_len'],
    'fragmented': gate_frag['realized_full_len'],
    'inflation': gate_frag['inflation_full'],
})

print('Evidence→readout gaps:', {
    'canonical': can_gap,
    'fragmented_normal': normal_gap,
    'block_affine': block_gap,
    'fractional': frac_gap,
})

if not normal_gap > can_gap:
    raise RuntimeError(
        'Normal fragmented positions did not inflate evidence→readout distance.'
    )

if abs(block_gap - can_gap) > 1e-4:
    raise RuntimeError(
        'Block-affine mapping did not restore canonical global gap.'
    )

if abs(frac_gap - can_gap) > 1e-4:
    raise RuntimeError(
        'Fractional mapping did not restore canonical global gap.'
    )

if (
    gate_frag['block_position_hash']
    == gate_frag['fractional_position_hash']
):
    raise RuntimeError(
        'Block-affine and fractional position mappings are identical.'
    )

scores = {}

scores['canonical_normal'] = score_chunked(
    gate_can['ids'],
    gate_can['normal_positions'],
    gate_inst.gold_digit,
)

scores['filler_frag_normal'] = score_chunked(
    gate_frag['ids'],
    gate_frag['normal_positions'],
    gate_inst.gold_digit,
)

scores['filler_frag_block_affine'] = score_chunked(
    gate_frag['ids'],
    gate_frag['block_affine_positions'],
    gate_inst.gold_digit,
)

scores['filler_frag_fractional'] = score_chunked(
    gate_frag['ids'],
    gate_frag['fractional_positions'],
    gate_inst.gold_digit,
)

for name, s in scores.items():
    print(
        name,
        {
            'pred': s['predicted_digit'],
            'forced_nll': s['forced_choice_nll'],
            'full_nll': s['gold_full_nll'],
            'full_argmax': s['full_argmax_text'],
            'peak_gb': s['peak_cuda_gb'],
        },
    )

    if not (
        np.isfinite(s['forced_choice_nll'])
        and np.isfinite(s['gold_full_nll'])
    ):
        raise RuntimeError(
            f'Non-finite score in {name}.'
        )

print('STRUCTURAL ENGINEERING GATE PASSED.')

## 13. Resumable JSONL utilities

In [ ]:
def load_jsonl(path):
    rows = []

    if path.exists():
        with path.open(
            'r',
            encoding='utf-8',
        ) as f:
            for line in f:
                line = line.strip()

                if line:
                    rows.append(
                        json.loads(line)
                    )

    return rows

def completed_ids(path):
    return {
        row['result_id']
        for row in load_jsonl(path)
        if 'result_id' in row
    }

def make_result_id(meta):
    blob = json.dumps(
        meta,
        sort_keys=True,
        separators=(',', ':'),
    ).encode('utf-8')

    return hashlib.sha256(
        blob
    ).hexdigest()[:24]

def append_jsonl(path, row):
    with path.open(
        'a',
        encoding='utf-8',
    ) as f:
        f.write(
            json.dumps(
                row,
                ensure_ascii=False,
            )
            + '\n'
        )
        f.flush()

# Phase A — central 4K/8K causal mechanism run

16 holdout semantic instances × 2 filler languages × 2 canonical lengths × 4 conditions = **256 rows**.

Condition execution order is deterministically shuffled within every base instance.

In [ ]:
CONDITION_SPECS = [
    (
        'canonical_normal',
        None,
        'normal_positions',
        'gaps_normal',
    ),
    (
        'filler_frag_normal',
        'filler',
        'normal_positions',
        'gaps_normal',
    ),
    (
        'filler_frag_block_affine',
        'filler',
        'block_affine_positions',
        'gaps_block_affine',
    ),
    (
        'filler_frag_fractional',
        'filler',
        'fractional_positions',
        'gaps_fractional',
    ),
]

done = completed_ids(
    RESULTS_PATH
)

for filler_language in FILLER_LANGUAGES:
    for target_chat_tokens in TARGET_LENGTHS:
        for trial_index, semantic_seed in enumerate(
            SEMANTIC_SEEDS
        ):
            inst = build_instance(
                filler_language=filler_language,
                semantic_seed=semantic_seed,
                trial_index=trial_index,
                target_chat_tokens=target_chat_tokens,
            )

            canonical_enc = assemble_chat_encoding(
                inst,
                fragment_target=None,
            )

            fragmented_enc = assemble_chat_encoding(
                inst,
                fragment_target='filler',
            )

            spec_order = CONDITION_SPECS[:]

            stable_rng(
                'condition_order_v060',
                filler_language,
                target_chat_tokens,
                semantic_seed,
                trial_index,
            ).shuffle(
                spec_order
            )

            evidence_text = '\n'.join(
                inst.evidence_lines
            )

            query_text = '\n'.join(
                inst.query_lines
            )

            for order_index, (
                condition,
                fragment_target,
                pos_key,
                gap_key,
            ) in enumerate(
                spec_order
            ):
                enc = (
                    canonical_enc
                    if fragment_target is None
                    else fragmented_enc
                )

                keep_pct = (
                    None
                    if fragment_target is None
                    else FRAG_KEEP_BY_LANGUAGE[
                        filler_language
                    ]
                )

                meta = {
                    'phase': 'mechanism_4k8k',
                    'model': MODEL_NAME,
                    'task_version': TASK_VERSION,
                    'scorer_version': SCORER_VERSION,
                    'position_intervention': POSITION_INTERVENTION,
                    'filler_language': filler_language,
                    'target_chat_tokens': target_chat_tokens,
                    'semantic_seed': semantic_seed,
                    'trial_index': trial_index,
                    'n_records': N_RECORDS,
                    'condition': condition,
                    'frag_keep': keep_pct,
                }

                rid = make_result_id(meta)

                if rid in done:
                    continue

                scored = score_chunked(
                    enc['ids'],
                    enc[pos_key],
                    inst.gold_digit,
                )

                gap = enc[gap_key]

                row = {
                    'result_id': rid,
                    **meta,
                    'condition_order_index': order_index,
                    'gold_digit': inst.gold_digit,
                    'target_key': inst.target_key,
                    'target_record_position': inst.target_record_position,
                    'canonical_full_tokens': enc[
                        'canonical_full_len'
                    ],
                    'realized_full_tokens': enc[
                        'realized_full_len'
                    ],
                    'canonical_user_tokens': enc[
                        'canonical_user_len'
                    ],
                    'realized_user_tokens': enc[
                        'realized_user_len'
                    ],
                    'inflation_full': enc[
                        'inflation_full'
                    ],
                    'inflation_user': enc[
                        'inflation_user'
                    ],
                    'position_max': float(
                        max(enc[pos_key])
                    ),
                    'evidence_query_position_gap': gap[
                        'evidence_query'
                    ],
                    'evidence_readout_position_gap': gap[
                        'evidence_readout'
                    ],
                    'filler_position_span': gap[
                        'filler_span'
                    ],
                    'query_readout_position_gap': gap[
                        'query_readout'
                    ],
                    'ids_hash': enc[
                        'ids_hash'
                    ],
                    'position_hash': (
                        enc[
                            'normal_position_hash'
                        ]
                        if pos_key == 'normal_positions'
                        else (
                            enc[
                                'block_position_hash'
                            ]
                            if pos_key
                            == 'block_affine_positions'
                            else enc[
                                'fractional_position_hash'
                            ]
                        )
                    ),
                    'prompt_sha256': hashlib.sha256(
                        inst.user_content.encode(
                            'utf-8'
                        )
                    ).hexdigest(),
                    'evidence_sha256': hashlib.sha256(
                        evidence_text.encode(
                            'utf-8'
                        )
                    ).hexdigest(),
                    'query_sha256': hashlib.sha256(
                        query_text.encode(
                            'utf-8'
                        )
                    ).hexdigest(),
                    'dial_fingerprint': (
                        d100.fingerprint
                        if fragment_target is None
                        else DIALS[
                            filler_language
                        ].fingerprint
                    ),
                    'timestamp_utc': (
                        pd.Timestamp.utcnow().isoformat()
                    ),
                    **flatten_score(
                        scored
                    ),
                }

                append_jsonl(
                    RESULTS_PATH,
                    row,
                )

                done.add(rid)

                print(
                    f"fill={filler_language} "
                    f"L={target_chat_tokens:5d} "
                    f"trial={trial_index:02d} "
                    f"{condition:<28} "
                    f"gold={inst.gold_digit} "
                    f"pred={scored['predicted_digit']} "
                    f"n={enc['realized_full_len']:5d} "
                    f"gapER={gap['evidence_readout']:8.2f} "
                    f"NLL={scored['forced_choice_nll']:.5f} "
                    f"acc={scored['correct']} "
                    f"full={scored['full_argmax_text']!r} "
                    f"peak={scored['peak_cuda_gb']:.2f}GB"
                )

print('Mechanism run complete:', RESULTS_PATH)

## 14. Pairing, token-ID, prompt, and gold-balance gates

In [ ]:
mech = pd.DataFrame(
    load_jsonl(
        RESULTS_PATH
    )
)

base_cols = [
    'filler_language',
    'target_chat_tokens',
    'semantic_seed',
    'trial_index',
]

condition_counts = mech.groupby(
    base_cols
)['condition'].nunique()

print(
    'Condition-count distribution:',
    condition_counts.value_counts().sort_index().to_dict(),
)

if (condition_counts != 4).any():
    display(
        condition_counts[
            condition_counts != 4
        ].reset_index(
            name='n_conditions'
        )
    )
    raise RuntimeError(
        'Incomplete mechanism base instances.'
    )

prompt_counts = mech.groupby(
    base_cols
)['prompt_sha256'].nunique()

if (prompt_counts != 1).any():
    raise RuntimeError(
        'Prompt text differs across conditions.'
    )

frag_names = [
    'filler_frag_normal',
    'filler_frag_block_affine',
    'filler_frag_fractional',
]

frag = mech[
    mech['condition'].isin(frag_names)
]

frag_id_counts = frag.groupby(
    base_cols
)['ids_hash'].nunique()

if (frag_id_counts != 1).any():
    raise RuntimeError(
        'Fragmented position conditions do not have identical token IDs.'
    )

task_pair_cols = [
    'target_chat_tokens',
    'semantic_seed',
    'trial_index',
]

if (
    mech.groupby(
        task_pair_cols
    )['evidence_sha256'].nunique()
    != 1
).any():
    raise RuntimeError(
        'Task evidence differs across filler languages.'
    )

if (
    mech.groupby(
        task_pair_cols
    )['query_sha256'].nunique()
    != 1
).any():
    raise RuntimeError(
        'Task query differs across filler languages.'
    )

canonical = mech[
    mech['condition']
    == 'canonical_normal'
]

gold_table = pd.crosstab(
    [
        canonical['filler_language'],
        canonical['target_chat_tokens'],
    ],
    canonical['gold_digit'],
).reindex(
    columns=DIGITS,
    fill_value=0,
)

display(gold_table)

if not (gold_table == 4).all().all():
    raise RuntimeError(
        'Gold digits are not exactly balanced in every language×length cell.'
    )

print(
    'PAIRING / SAME-ID / TASK-INVARIANCE / GOLD-BALANCE GATES PASSED.'
)

## 15. Manipulation geometry audit

In [ ]:
central = [
    'canonical_normal',
    'filler_frag_normal',
    'filler_frag_block_affine',
    'filler_frag_fractional',
]

geometry = mech.groupby(
    [
        'filler_language',
        'target_chat_tokens',
        'condition',
    ],
    as_index=False,
).agg(
    n=('correct', 'size'),
    mean_tokens=('realized_full_tokens', 'mean'),
    min_tokens=('realized_full_tokens', 'min'),
    max_tokens=('realized_full_tokens', 'max'),
    mean_inflation=('inflation_full', 'mean'),
    mean_evidence_query_gap=(
        'evidence_query_position_gap',
        'mean',
    ),
    mean_evidence_readout_gap=(
        'evidence_readout_position_gap',
        'mean',
    ),
    mean_filler_span=('filler_position_span', 'mean'),
    mean_position_max=('position_max', 'mean'),
    max_peak_cuda_gb=('peak_cuda_gb', 'max'),
)

display(geometry)

gap_wide = mech.pivot_table(
    index=base_cols,
    columns='condition',
    values='evidence_readout_position_gap',
    aggfunc='first',
).reset_index()

gap_wide['normal_gap_inflation'] = (
    gap_wide['filler_frag_normal']
    - gap_wide['canonical_normal']
)

gap_wide['block_gap_error'] = (
    gap_wide['filler_frag_block_affine']
    - gap_wide['canonical_normal']
)

gap_wide['fractional_gap_error'] = (
    gap_wide['filler_frag_fractional']
    - gap_wide['canonical_normal']
)

display(
    gap_wide.groupby(
        [
            'filler_language',
            'target_chat_tokens',
        ],
        as_index=False,
    ).agg(
        mean_gap_inflation=(
            'normal_gap_inflation',
            'mean',
        ),
        min_gap_inflation=(
            'normal_gap_inflation',
            'min',
        ),
        max_abs_block_error=(
            'block_gap_error',
            lambda s: float(
                np.abs(s).max()
            ),
        ),
        max_abs_fractional_error=(
            'fractional_gap_error',
            lambda s: float(
                np.abs(s).max()
            ),
        ),
    )
)

if not (
    gap_wide[
        'normal_gap_inflation'
    ] > 0
).all():
    raise RuntimeError(
        'Some fragmented-normal rows did not inflate positional distance.'
    )

if (
    np.abs(
        gap_wide[
            'block_gap_error'
        ]
    ).max()
    > 1e-3
):
    raise RuntimeError(
        'Block-affine mapping did not restore canonical global distance.'
    )

if (
    np.abs(
        gap_wide[
            'fractional_gap_error'
        ]
    ).max()
    > 1e-3
):
    raise RuntimeError(
        'Fractional mapping did not restore canonical global distance.'
    )

# Same global span but distinct internal mapping.
pos_pair = mech[
    mech['condition'].isin(
        [
            'filler_frag_block_affine',
            'filler_frag_fractional',
        ]
    )
].pivot_table(
    index=base_cols,
    columns='condition',
    values='position_hash',
    aggfunc='first',
).reset_index()

if (
    pos_pair[
        'filler_frag_block_affine'
    ]
    == pos_pair[
        'filler_frag_fractional'
    ]
).any():
    raise RuntimeError(
        'Block-affine and fractional position arrays unexpectedly match.'
    )

print('MANIPULATION GEOMETRY GATE PASSED.')

## 16. Performance table

In [ ]:
performance = mech.groupby(
    [
        'filler_language',
        'target_chat_tokens',
        'condition',
    ],
    as_index=False,
).agg(
    n=('correct', 'size'),
    accuracy=('correct', 'mean'),
    mean_forced_nll=('forced_choice_nll', 'mean'),
    median_forced_nll=('forced_choice_nll', 'median'),
    mean_full_nll=('gold_full_nll', 'mean'),
    mean_gold_probability=(
        'gold_candidate_probability',
        'mean',
    ),
    full_argmax_gold_rate=(
        'full_argmax_is_gold',
        'mean',
    ),
    full_argmax_legal_rate=(
        'full_argmax_is_legal_digit',
        'mean',
    ),
    mean_tokens=('realized_full_tokens', 'mean'),
    mean_position_gap=(
        'evidence_readout_position_gap',
        'mean',
    ),
    max_peak_cuda_gb=('peak_cuda_gb', 'max'),
    mean_seconds=('seconds', 'mean'),
)

display(performance)

# Paired causal effects

## 17. Effect construction

Positive values mean:

- **fragmentation_penalty**: fragmentation made retrieval worse;
- **block_affine_rescue**: generic compression recovered loss;
- **fractional_rescue**: source-aligned positions recovered loss;
- **fractional_advantage_over_block**: source alignment outperformed generic compression.

`fractional_residual_vs_canonical` is the remaining non-positional fragmentation effect after source-aligned rescue. Values near zero mean the filler penalty was almost entirely positional.

In [ ]:
def make_effect_table(
    df,
    metric='forced_choice_nll',
):
    conditions = [
        'canonical_normal',
        'filler_frag_normal',
        'filler_frag_block_affine',
        'filler_frag_fractional',
    ]

    wide = df[
        df['condition'].isin(
            conditions
        )
    ].pivot_table(
        index=base_cols,
        columns='condition',
        values=metric,
        aggfunc='first',
    ).reset_index()

    wide = wide.dropna(
        subset=conditions
    ).copy()

    wide['fragmentation_penalty'] = (
        wide['filler_frag_normal']
        - wide['canonical_normal']
    )

    wide['block_affine_rescue'] = (
        wide['filler_frag_normal']
        - wide['filler_frag_block_affine']
    )

    wide['fractional_rescue'] = (
        wide['filler_frag_normal']
        - wide['filler_frag_fractional']
    )

    wide['fractional_advantage_over_block'] = (
        wide['filler_frag_block_affine']
        - wide['filler_frag_fractional']
    )

    wide['fractional_residual_vs_canonical'] = (
        wide['filler_frag_fractional']
        - wide['canonical_normal']
    )

    return wide

forced_effects = make_effect_table(
    mech,
    'forced_choice_nll',
)

full_effects = make_effect_table(
    mech,
    'gold_full_nll',
)

display(forced_effects.head(20))

## 18. Cluster-aware bootstrap confidence intervals

For pooled estimates, the bootstrap resamples **semantic trials as clusters** so the four repeated language×length observations for a trial stay together.

Cell-specific intervals resample trials within that cell.

In [ ]:
EFFECT_NAMES = [
    'fragmentation_penalty',
    'block_affine_rescue',
    'fractional_rescue',
    'fractional_advantage_over_block',
    'fractional_residual_vs_canonical',
]

def bootstrap_mean_ci(
    values,
    n_boot=BOOTSTRAP_REPS,
    seed=BOOTSTRAP_SEED,
):
    x = np.asarray(
        pd.Series(values).dropna(),
        dtype=float,
    )

    if len(x) == 0:
        return np.nan, np.nan, np.nan

    if len(x) == 1:
        return float(x[0]), np.nan, np.nan

    rng = np.random.default_rng(seed)

    idx = rng.integers(
        0,
        len(x),
        size=(n_boot, len(x)),
    )

    boot = x[idx].mean(axis=1)

    return (
        float(x.mean()),
        float(np.quantile(boot, 0.025)),
        float(np.quantile(boot, 0.975)),
    )

def cluster_bootstrap_mean_ci(
    df,
    value_col,
    cluster_col='trial_index',
    n_boot=BOOTSTRAP_REPS,
    seed=BOOTSTRAP_SEED,
):
    clean = df[
        [
            cluster_col,
            value_col,
        ]
    ].dropna().copy()

    clusters = np.asarray(
        sorted(
            clean[cluster_col].unique()
        )
    )

    if len(clusters) == 0:
        return np.nan, np.nan, np.nan

    observed = float(
        clean[value_col].mean()
    )

    if len(clusters) == 1:
        return observed, np.nan, np.nan

    grouped = {
        c: clean.loc[
            clean[cluster_col] == c,
            value_col,
        ].to_numpy(dtype=float)
        for c in clusters
    }

    rng = np.random.default_rng(seed)
    boot_means = np.empty(n_boot)

    for b in range(n_boot):
        sampled = rng.choice(
            clusters,
            size=len(clusters),
            replace=True,
        )

        vals = np.concatenate(
            [
                grouped[c]
                for c in sampled
            ]
        )

        boot_means[b] = vals.mean()

    return (
        observed,
        float(np.quantile(boot_means, 0.025)),
        float(np.quantile(boot_means, 0.975)),
    )

cell_rows = []

for (
    filler_language,
    target_chat_tokens
), g in forced_effects.groupby(
    [
        'filler_language',
        'target_chat_tokens',
    ]
):
    for effect in EFFECT_NAMES:
        mean, lo, hi = bootstrap_mean_ci(
            g[effect],
            seed=(
                BOOTSTRAP_SEED
                + stable_int(
                    filler_language,
                    target_chat_tokens,
                    effect,
                )
                % 100000
            ),
        )

        cell_rows.append({
            'filler_language': filler_language,
            'target_chat_tokens': int(
                target_chat_tokens
            ),
            'effect': effect,
            'n': len(g),
            'mean': mean,
            'ci_lo': lo,
            'ci_hi': hi,
            'positive_n': int(
                (g[effect] > 0).sum()
            ),
            'negative_n': int(
                (g[effect] < 0).sum()
            ),
        })

cell_effect_summary = pd.DataFrame(
    cell_rows
)

display(cell_effect_summary)

pooled_rows = []

for metric_name, eff_df in [
    ('forced_choice_nll', forced_effects),
    ('gold_full_nll', full_effects),
]:
    for effect in EFFECT_NAMES:
        mean, lo, hi = cluster_bootstrap_mean_ci(
            eff_df,
            effect,
            seed=(
                BOOTSTRAP_SEED
                + stable_int(
                    metric_name,
                    effect,
                )
                % 100000
            ),
        )

        pooled_rows.append({
            'metric': metric_name,
            'effect': effect,
            'n_rows': len(eff_df),
            'n_trial_clusters': eff_df[
                'trial_index'
            ].nunique(),
            'mean': mean,
            'ci_lo': lo,
            'ci_hi': hi,
            'positive_n': int(
                (eff_df[effect] > 0).sum()
            ),
            'negative_n': int(
                (eff_df[effect] < 0).sum()
            ),
        })

pooled_effect_summary = pd.DataFrame(
    pooled_rows
)

display(pooled_effect_summary)

print('\nPRIMARY: FORCED-CHOICE NLL')
for _, row in pooled_effect_summary[
    pooled_effect_summary['metric']
    == 'forced_choice_nll'
].iterrows():
    print(
        f"{row['effect']:<38} "
        f"mean={row['mean']:+.6f} "
        f"95% cluster-CI=[{row['ci_lo']:+.6f},{row['ci_hi']:+.6f}] "
        f"positive={int(row['positive_n'])}/{int(row['n_rows'])}"
    )

## 19. Rescue fractions and language/length diagnostics

Rescue fractions are reported only when the corresponding mean fragmentation penalty is positive.

They are descriptive, not bounded to `[0,1]`: values above 1 indicate over-recovery.

In [ ]:
diag_rows = []

for (
    filler_language,
    target_chat_tokens
), g in forced_effects.groupby(
    [
        'filler_language',
        'target_chat_tokens',
    ]
):
    frag_pen = float(
        g['fragmentation_penalty'].mean()
    )

    block = float(
        g['block_affine_rescue'].mean()
    )

    frac = float(
        g['fractional_rescue'].mean()
    )

    diag_rows.append({
        'filler_language': filler_language,
        'target_chat_tokens': int(
            target_chat_tokens
        ),
        'fragmentation_penalty': frag_pen,
        'block_affine_rescue': block,
        'fractional_rescue': frac,
        'block_rescue_fraction': (
            block / frag_pen
            if frag_pen > 0
            else np.nan
        ),
        'fractional_rescue_fraction': (
            frac / frag_pen
            if frag_pen > 0
            else np.nan
        ),
        'fractional_advantage_over_block': float(
            g[
                'fractional_advantage_over_block'
            ].mean()
        ),
        'fractional_residual_vs_canonical': float(
            g[
                'fractional_residual_vs_canonical'
            ].mean()
        ),
    })

diagnostics = pd.DataFrame(
    diag_rows
)

display(diagnostics)

# Mechanism decision

## 20. Directional mechanism gate

This gate decides whether the mechanism is coherent enough to justify the next decomposition/replication stage. It is **not** a replacement for the confidence intervals above.

Requirements:

- canonical forced-choice accuracy ≥ 90%;
- canonical predictions are not collapsed to one digit;
- fragmentation penalty is positive pooled and in at least 3/4 language×length cells;
- at least one positional rescue is positive pooled and in at least 3/4 cells;
- all structural geometry gates have already passed.

`SOURCE_STRUCTURE_SIGNAL` is separate: it requires fractional source alignment to outperform block-affine compression directionally.

In [ ]:
canonical = mech[
    mech['condition']
    == 'canonical_normal'
].copy()

canonical_acc = float(
    canonical['correct'].mean()
)

canonical_pred_counts = (
    canonical['predicted_digit']
    .value_counts()
    .reindex(
        DIGITS,
        fill_value=0,
    )
)

canonical_max_pred_share = float(
    canonical_pred_counts.max()
    / canonical_pred_counts.sum()
)

primary_pooled = (
    pooled_effect_summary[
        pooled_effect_summary['metric']
        == 'forced_choice_nll'
    ]
    .set_index('effect')
)

pooled = {
    effect: float(
        primary_pooled.loc[
            effect,
            'mean',
        ]
    )
    for effect in EFFECT_NAMES
}

cell_wide = cell_effect_summary.pivot_table(
    index=[
        'filler_language',
        'target_chat_tokens',
    ],
    columns='effect',
    values='mean',
    aggfunc='first',
).reset_index()

display(cell_wide)

frag_positive_cells = int(
    (
        cell_wide[
            'fragmentation_penalty'
        ] > 0
    ).sum()
)

block_positive_cells = int(
    (
        cell_wide[
            'block_affine_rescue'
        ] > 0
    ).sum()
)

frac_positive_cells = int(
    (
        cell_wide[
            'fractional_rescue'
        ] > 0
    ).sum()
)

source_positive_cells = int(
    (
        cell_wide[
            'fractional_advantage_over_block'
        ] > 0
    ).sum()
)

best_rescue = max(
    pooled['block_affine_rescue'],
    pooled['fractional_rescue'],
)

best_rescue_cells = max(
    block_positive_cells,
    frac_positive_cells,
)

POSITIONAL_SIGNAL = (
    pooled['fragmentation_penalty'] > 0
    and best_rescue > 0
    and frag_positive_cells >= 3
    and best_rescue_cells >= 3
)

SOURCE_STRUCTURE_SIGNAL = (
    pooled[
        'fractional_advantage_over_block'
    ] > 0
    and source_positive_cells >= 3
)

MECHANISM_GO = (
    canonical_acc >= 0.90
    and canonical_max_pred_share <= 0.40
    and POSITIONAL_SIGNAL
)

print('Canonical forced-choice accuracy:', canonical_acc)
print(
    'Canonical prediction counts:',
    canonical_pred_counts.to_dict(),
)
print(
    'Canonical max prediction share:',
    canonical_max_pred_share,
)
print(
    'Pooled primary NLL effects:',
    json.dumps(
        pooled,
        indent=2,
    ),
)
print(
    'Positive language×length cells:',
    {
        'fragmentation_penalty': frag_positive_cells,
        'block_affine_rescue': block_positive_cells,
        'fractional_rescue': frac_positive_cells,
        'fractional_advantage_over_block': source_positive_cells,
    },
)

print('\nPOSITIONAL_SIGNAL =', POSITIONAL_SIGNAL)
print(
    'SOURCE_STRUCTURE_SIGNAL =',
    SOURCE_STRUCTURE_SIGNAL,
)
print('MECHANISM_GO =', MECHANISM_GO)

if MECHANISM_GO:
    print(
        '\nPASS: filler fragmentation shows a coherent positional mechanism. '
        'Review effect sizes/CIs before running the evidence-locus decomposition.'
    )
else:
    print(
        '\nSTOP: do not add evidence fragmentation or another model yet. '
        'Diagnose the central causal contrast first.'
    )

## 21. Save final audit

In [ ]:
primary_ci = {}

for effect in EFFECT_NAMES:
    row = primary_pooled.loc[
        effect
    ]

    primary_ci[
        effect
    ] = {
        'mean': float(
            row['mean']
        ),
        'ci_lo': float(
            row['ci_lo']
        ),
        'ci_hi': float(
            row['ci_hi']
        ),
    }

audit = {
    'model': MODEL_NAME,
    'transformers_version': transformers.__version__,
    'notebook_version': 'v0.6.0',
    'task_version': TASK_VERSION,
    'claim_scope': 'associative_retrieval',
    'scorer_version': SCORER_VERSION,
    'position_intervention': POSITION_INTERVENTION,
    'calibration_audit_path': str(
        CALIBRATION_AUDIT
    ),
    'frozen_n_records': N_RECORDS,
    'frozen_frag_keep_by_language': FRAG_KEEP_BY_LANGUAGE,
    'holdout_semantic_seeds': SEMANTIC_SEEDS,
    'target_lengths': TARGET_LENGTHS,
    'primary_metric': 'forced_choice_nll',
    'secondary_metric': 'gold_full_nll',
    'same_fragmented_token_ids_across_position_conditions': True,
    'task_relevant_evidence_language_invariant': True,
    'language_factor': 'irrelevant_filler_language',
    'canonical_accuracy': canonical_acc,
    'canonical_max_pred_share': canonical_max_pred_share,
    'primary_pooled_effects_cluster_bootstrap': primary_ci,
    'positive_cells': {
        'fragmentation_penalty': frag_positive_cells,
        'block_affine_rescue': block_positive_cells,
        'fractional_rescue': frac_positive_cells,
        'fractional_advantage_over_block': source_positive_cells,
    },
    'positional_signal': bool(
        POSITIONAL_SIGNAL
    ),
    'source_structure_signal': bool(
        SOURCE_STRUCTURE_SIGNAL
    ),
    'mechanism_go': bool(
        MECHANISM_GO
    ),
}

with AUDIT_PATH.open(
    'w',
    encoding='utf-8',
) as f:
    json.dump(
        audit,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    )
)

print('Saved:', AUDIT_PATH)

# Files to upload

From:

`MyDrive/qwen3_assoc_mechanism_v060/`

upload:

1. `v060_mechanism_4k8k.jsonl`
2. `v060_audit.json`

Key console lines:

- `POSITIONAL_SIGNAL = True/False`
- `SOURCE_STRUCTURE_SIGNAL = True/False`
- `MECHANISM_GO = True/False`

Do **not** add evidence-locus fragmentation, 16K, or a second model until these central causal results are reviewed.